# Ministral 3B Unified Fine-tuning for TravelOrderResolver

This notebook fine-tunes Ministral 3B to handle the entire NLP pipeline in one inference:
- **Language Detection** (fr/en/other)
- **Intent Classification** (TRIP/NOT_TRIP/UNKNOWN)
- **Entity Extraction** (departure, destination, intermediate)

**Output format**: JSON with all fields in one response

**Requirements**: Free Colab T4 GPU (16GB VRAM)

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
# Get latest transformers for Ministral support
!pip install --upgrade transformers trl datasets accelerate

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Ministral 3B with Unsloth

In [ ]:
from unsloth import FastVisionModel
import torch

# Load model with runtime 4-bit quantization (NOT pre-quantized)
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Ministral-3-3B-Instruct-2512",
    max_seq_length=512,
    load_in_4bit=True,  # Runtime quantization - avoids PEFT compatibility issues
    dtype=None,  # Auto-detect
)

# Setup padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully!")

## 3. Apply LoRA

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # Text-only task, skip vision
    finetune_language_layers=True,     # Fine-tune language layers
    finetune_attention_modules=True,   # Include attention
    finetune_mlp_modules=True,         # Include MLP
    r=32,                              # LoRA rank
    lora_alpha=32,                     # LoRA alpha (same as rank)
    lora_dropout=0,                    # MUST be 0 for Unsloth
    bias="none",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

## 4. Upload Training Data

Upload `train.json` and `val.json` from `datasets/base/`

In [ ]:
from google.colab import files

print("Please upload train.json and val.json from datasets/base/")
uploaded = files.upload()

## 5. Prepare Unified Dataset

In [ ]:
import json
from datasets import Dataset

# Unified prompt template
UNIFIED_PROMPT = """[INST] Analyse cette phrase de voyage.
Reponds en JSON avec: langue (fr/en/other), intention (TRIP/NOT_TRIP/UNKNOWN),
et si TRIP: depart, destination, intermediaires.

Phrase: {text} [/INST]"""

def convert_to_unified(sample):
    """Convert a sample to unified prompt+response format."""
    prompt = UNIFIED_PROMPT.format(text=sample["sentence"])
    
    # Map language field
    lang = "fr" if sample.get("language", "FRENCH") == "FRENCH" else "other"
    
    # Parse intermediate stops
    intermediate = sample.get("intermediate", "")
    if intermediate and isinstance(intermediate, str) and intermediate.strip():
        intermediaires = [s.strip() for s in intermediate.split(",") if s.strip()]
    else:
        intermediaires = []
    
    # Build target JSON
    target = {
        "langue": lang,
        "intention": sample["intent"],
        "depart": sample.get("departure") or None,
        "destination": sample.get("destination") or None,
        "intermediaires": intermediaires
    }
    
    # Full training text: prompt + response + EOS
    full_text = prompt + json.dumps(target, ensure_ascii=False) + tokenizer.eos_token
    return full_text

def load_and_convert(filename):
    """Load JSON file and convert to unified format."""
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    texts = [convert_to_unified(sample) for sample in data]
    return Dataset.from_dict({"text": texts})

# Load datasets
train_dataset = load_and_convert("train.json")
val_dataset = load_and_convert("val.json")

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"\nExample:\n{train_dataset[0]['text'][:500]}...")

## 6. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="ministral-unified-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.001,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="adamw_8bit",  # Unsloth recommended
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=512,
)

print("Starting training...")

In [ ]:
# Train!
trainer.train()

## 7. Test the Model

In [ ]:
# Set to inference mode
FastVisionModel.for_inference(model)

def test_inference(text):
    """Test the unified model."""
    prompt = UNIFIED_PROMPT.format(text=text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the generated part (after [/INST])
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()
    return response

# Test cases
test_cases = [
    "Je voudrais aller de Paris a Lyon",
    "Je veux aller de Marseille a Nice en passant par Toulon",
    "Quel temps fait-il demain?",
    "Commande un taxi pour moi",
]

print("=" * 60)
print("Model Test Results")
print("=" * 60)

for text in test_cases:
    result = test_inference(text)
    print(f"\nInput: {text}")
    print(f"Output: {result}")
    print("-" * 40)

## 8. Save and Download Adapter

In [ ]:
# Save LoRA adapter
model.save_pretrained("ministral-unified-lora")
tokenizer.save_pretrained("ministral-unified-lora")

print("Adapter saved!")
!ls -la ministral-unified-lora/

In [ ]:
# Zip and download
!zip -r ministral-unified-lora.zip ministral-unified-lora/
print(f"\nZip size: {!ls -lh ministral-unified-lora.zip}")

files.download("ministral-unified-lora.zip")

## 9. Local Usage Instructions

After downloading `ministral-unified-lora.zip`:

```bash
# 1. Unzip the adapter
unzip ministral-unified-lora.zip -d models/

# 2. Test locally
python -c "
from src.nlp.unified.ministral_unified import MinistralUnifiedNLP
nlp = MinistralUnifiedNLP('models/ministral-unified-lora')
print(nlp.process('Je voudrais aller de Paris a Lyon'))
"
```

Expected output:
```json
{"langue": "fr", "intention": "TRIP", "depart": "Paris", "destination": "Lyon", "intermediaires": []}
```